# Chimera extension

In [1]:
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
plt.rcParams["axes.grid"] = False
import numpy as np
import pandas as pd

from gen_cats.evaluation.report_analysis import (
    CHIMERA_DIR,
    chimera_sample_paths,
    ensure_plots_dir,
    write_stats_csv,
)

PLOTS = ensure_plots_dir("results")
paths = chimera_sample_paths(CHIMERA_DIR)
paths

[PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/checkpoints/chimera/wgan_gp/chimera_64_seed0/additional_samples.png'),
 PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/checkpoints/chimera/wgan_gp/chimera_64_seed3407/additional_samples.png'),
 PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/checkpoints/chimera/wgan_gp/chimera_64_seed42/additional_samples.png')]

In [2]:
n_seeds = len(paths)
fig_width = 10.5
panel_width = fig_width / n_seeds
title_space = 0.22
fig_height = panel_width + title_space

fig, axes = plt.subplots(1, n_seeds, figsize=(fig_width, fig_height))
if n_seeds == 1:
    axes = [axes]

for ax, path in zip(axes, paths, strict=True):
    ax.imshow(mpimg.imread(path), aspect="equal")
    ax.set_title(path.parent.name, fontsize=10, pad=4)
    ax.axis("off")

fig.subplots_adjust(wspace=0.03, left=0.005, right=0.995, top=0.90, bottom=0.005)
out = PLOTS / "chimera_panel.png"
fig.savefig(out, dpi=200, bbox_inches="tight", pad_inches=0.01)
plt.close(fig)
out

PosixPath('/Users/pawelp/Desktop/education/pw/deepl/gen-cats/notebooks/plots/results/chimera_panel.png')

In [3]:
SAMPLE_ROWS = 2
SAMPLES_PER_ROW = 4
EPOCH_COLS = 4


def split_sample_grid(path, grid: int = 4) -> list[np.ndarray]:
    img = mpimg.imread(path)
    h, w = img.shape[:2]
    crop = (min(h, w) // grid) * grid
    off_y = (h - crop) // 2
    off_x = (w - crop) // 2
    img = img[off_y : off_y + crop, off_x : off_x + crop]
    tile_h, tile_w = crop // grid, crop // grid
    tiles: list[np.ndarray] = []
    for row in range(grid):
        for col in range(grid):
            tiles.append(
                img[row * tile_h : (row + 1) * tile_h, col * tile_w : (col + 1) * tile_w]
            )
    return tiles


def stitch_tiles(tiles: list[np.ndarray], rows: int, cols: int, gap: int = 2) -> np.ndarray:
    tile_h, tile_w = tiles[0].shape[:2]
    channels = tiles[0].shape[2] if tiles[0].ndim == 3 else 1
    canvas = np.ones(
        (
            rows * tile_h + (rows - 1) * gap,
            cols * tile_w + (cols - 1) * gap,
            channels,
        ),
        dtype=tiles[0].dtype,
    )
    for row in range(rows):
        for col in range(cols):
            tile_i = row * cols + col
            if tile_i >= len(tiles):
                continue
            y0 = row * (tile_h + gap)
            x0 = col * (tile_w + gap)
            canvas[y0 : y0 + tile_h, x0 : x0 + tile_w] = tiles[tile_i]
    return canvas


def plot_chimera_progress(seed_dir, outfile, max_frames: int = 8, epoch_cols: int = EPOCH_COLS) -> None:
    epoch_pngs = sorted(
        seed_dir.glob("samples_epoch*.png"),
        key=lambda p: int(p.stem.replace("samples_epoch", "")),
    )
    if not epoch_pngs:
        return
    if len(epoch_pngs) > max_frames:
        idx = [round(i * (len(epoch_pngs) - 1) / (max_frames - 1)) for i in range(max_frames)]
        epoch_pngs = [epoch_pngs[i] for i in idx]

    n_epochs = len(epoch_pngs)
    epoch_rows = (n_epochs + epoch_cols - 1) // epoch_cols
    fig_width = 10.5
    panel_width = fig_width / epoch_cols
    panel_height = panel_width * (SAMPLE_ROWS / SAMPLES_PER_ROW)
    epoch_title_space = 0.16
    fig_height = 0.22 + epoch_rows * (panel_height + epoch_title_space) + 0.04

    fig = plt.figure(figsize=(fig_width, fig_height))
    gs = fig.add_gridspec(
        epoch_rows + 1,
        epoch_cols,
        height_ratios=[0.14] + [1.0] * epoch_rows,
        hspace=0.24,
        wspace=0.05,
    )

    header_ax = fig.add_subplot(gs[0, :])
    header_ax.axis("off")
    header_ax.text(
        0.5,
        0.5,
        f"Training progression — {seed_dir.name}",
        ha="center",
        va="center",
        fontsize=11,
        transform=header_ax.transAxes,
    )

    for epoch_i, path in enumerate(epoch_pngs):
        row, col = divmod(epoch_i, epoch_cols)
        ax = fig.add_subplot(gs[row + 1, col])
        tiles = split_sample_grid(path)[: SAMPLE_ROWS * SAMPLES_PER_ROW]
        ax.imshow(stitch_tiles(tiles, SAMPLE_ROWS, SAMPLES_PER_ROW), aspect="equal")
        ax.set_title(path.stem.replace("samples_", ""), fontsize=9, pad=3, loc="left")
        ax.axis("off")

    for j in range(n_epochs, epoch_rows * epoch_cols):
        row, col = divmod(j, epoch_cols)
        fig.add_subplot(gs[row + 1, col]).axis("off")

    fig.subplots_adjust(left=0.01, right=0.99, top=0.99, bottom=0.01)
    fig.savefig(outfile, dpi=200, bbox_inches="tight", pad_inches=0.01)
    plt.close(fig)


for path in paths:
    plot_chimera_progress(path.parent, PLOTS / f"chimera_progress_{path.parent.name}.png")